# Phutball zero-shot benchmark: Qwen3.5-9B (vLLM)

The same puzzle families the phutball network trained on, as text (`llm_bench/`): **win recognition** (can the side to
move win this turn? 200 positives with 1-4-jump wins + 200 engine-proven near-miss negatives) and the **unstoppable
threat** placement (200 puzzles; the answer set is complete). Every answer is checked by replaying it on the engine.

Plus three placement tasks from the same training families: **forced** (unstoppable threat), **block** (stop a
threatened winning jump when no defending jump exists) and **prevent** (stop an unstoppable threat one turn early, by a
placement or a jump sequence); 200 each.

Runs: instruct with thinking off, instruct with thinking on, and the base model (raw completion). Results go to
`phutball/llm_bench/` on Drive. Use an A100 80 GB or H100 runtime; disconnect when done.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/phutball/llm_bench'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U vllm 2>&1 | tail -2
!python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
import subprocess, time, urllib.request
def serve(model, max_len=32768):
    !pkill -f "vllm serve" ; sleep 5
    log = open(f'/content/vllm_{model.split("/")[-1]}.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', model, '--max-model-len', str(max_len), '--gpu-memory-utilization', '0.90',
                          '--port', '8000'], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):                                   # up to 15 min (download + load)
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', model); return p
        except Exception: time.sleep(5)
    raise RuntimeError('vLLM did not start: see ' + log.name)

def bench(model, name, extra=''):
    !python -m llm_bench.run --base-url http://localhost:8000/v1 --model {model} --name {name} --concurrency 48 {extra}
    !cp llm_bench/results/{name}.jsonl llm_bench/results/{name}.summary.json {OUT}/

In [ ]:
# 1. instruct, thinking OFF
M = 'Qwen/Qwen3.5-9B'
server = serve(M)
bench(M, 'qwen3.5-9b-nothink', '--thinking false --max-tokens 4096')

In [ ]:
# 2. instruct, thinking ON (long replies: the most expensive run)
bench(M, 'qwen3.5-9b-think', '--thinking true --max-tokens 24000 --timeout 1800')

In [ ]:
# 3. base model, raw completion (no chat template)
B = 'Qwen/Qwen3.5-9B-Base'
server = serve(B)
bench(B, 'qwen3.5-9b-base', '--completions --max-tokens 2048')

## 4. Budget forcing for truncated replies (~3 units)
A reply cut off at the token cap has no ANSWER line and scores as wrong. This re-sends each truncated reply with
"Time is up ... ANSWER:" appended and lets the model finish the answer line (30 tokens), so those replies are scored as
"best answer within the budget". It needs no new generation, only prefill of the saved replies.
Run this cell after the three runs above, or on its own in a fresh session (run the Drive / clone / serve() cells first).

In [ ]:
!mkdir -p llm_bench/results && cp {OUT}/qwen3.5-9b-*.jsonl llm_bench/results/
!git pull -q
server = serve('Qwen/Qwen3.5-9B')
!python -m llm_bench.run --model Qwen/Qwen3.5-9B --thinking false --salvage qwen3.5-9b-nothink --concurrency 32
!python -m llm_bench.run --model Qwen/Qwen3.5-9B --thinking true --salvage qwen3.5-9b-think --concurrency 16
server = serve('Qwen/Qwen3.5-9B-Base')
!python -m llm_bench.run --model Qwen/Qwen3.5-9B-Base --completions --salvage qwen3.5-9b-base --concurrency 32
!cp llm_bench/results/qwen3.5-9b-*.jsonl llm_bench/results/*-forced.summary.json {OUT}/

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{OUT}/*.summary.json')):
    s = json.load(open(f)); print(f.split('/')[-1])
    for k in ('win (all)', 'win: near-miss negative', 'win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump',
              'win: positive, 4-jump', 'forced placement', 'block placement', 'prevent placement'):
        if k in s: print(f'   {k:28s} {s[k]["accuracy"]:6.1%}  truncated {s[k].get("truncated", float("nan")):6.1%}  (n={s[k]["n"]})')
    print('   mean completion tokens', s.get('mean_completion_tokens'))
!pkill -f "vllm serve" 